# Imports, config, session and utilities

In [1]:
import json
from pathlib import Path
import sys
current_file_path = Path().resolve()
sys.path.insert(0, str(current_file_path.parent))
# These import statement after adding parent file path to recognize src python files
from src.support_agent.config import get_settings
from src.support_agent.snowflake_client import create_snowpark_session
# Getting .env config and init snowflake session
settings = get_settings()
session = create_snowpark_session(settings)

# Main code

In [2]:
query = """ SELECT * FROM PROJECT_DB.EVAL.RESULTS """

results_df = session.sql(query).to_pandas()
results_df["total_output_tokens"] = results_df["total_output_tokens"].apply(json.loads).apply(lambda d: d["total"])
results_df.shape

(42, 21)

In [17]:
results_df.columns

Index(['ticket_id', 'language', 'priority', 'type', 'query', 'ground_truth',
       'agent_answer', 'context', 'used_search', 'agent_model',
       'total_input_tokens', 'total_output_tokens', 'faithfulness_score',
       'faithfulness_explanation', 'answer_relevancy_score',
       'answer_relevancy_explanation', 'context_precision_score',
       'context_precision_explanation', 'context_recall_score',
       'context_recall_explanation', 'average_score'],
      dtype='object')

In [26]:
print("\n" + "=" * 80)
print("CHECK REASONING ")
print("=" * 80)
print(f"Total tickets found with similar problems in vector database: {results_df.used_search.value_counts()}")


CHECK REASONING 
Total tickets found with similar problems in vector database: used_search
False    3
Name: count, dtype: int64


In [9]:
# Print summary statistics
print("\n" + "=" * 80)
print("EVALUATION SUMMARY")
print("=" * 80)
print(f"Total tickets evaluated: {len(results_df)}")
print("\nAverage Scores:")
print(
    f"  Faithfulness:      {results_df['faithfulness_score'].mean():.3f}"
)
print(
    f"  Answer Relevancy:  {results_df['answer_relevancy_score'].mean():.3f}"
)
print(
    f"  Context Precision: {results_df['context_precision_score'].mean():.3f}"
)
print(
    f"  Context Recall:    {results_df['context_recall_score'].mean():.3f}"
)
print(
    f"  Overall Average:   {results_df['average_score'].mean():.3f}"
)

if "language" in results_df.columns:
    print("\Average score by Language:")
    print(results_df.groupby("language")["average_score"].mean())

if "priority" in results_df.columns:
    print("\Average score by Priority:")
    print(results_df.groupby("priority")["average_score"].mean())


EVALUATION SUMMARY
Total tickets evaluated: 3

Average Scores:
  Faithfulness:      0.000
  Answer Relevancy:  0.733
  Context Precision: 0.000
  Context Recall:    0.000
  Overall Average:   0.183
\Average score by Language:
language
de    0.2250
en    0.1625
Name: average_score, dtype: float64
\Average score by Priority:
priority
medium    0.183333
Name: average_score, dtype: float64


In [22]:
# Print summary statistics
print("\n" + "=" * 80)
print("USAGE SUMMARY")
print("=" * 80)
print(f"  Input tokens usage:      {results_df['total_input_tokens'].mean():.3f}")
print(f"  Output tokens usage:  {results_df['total_output_tokens'].mean():.3f}")


if "language" in results_df.columns:
    print("Average score by Language:")
    print(results_df.groupby("language")["total_input_tokens"].mean())

if "priority" in results_df.columns:
    print("Average score by Priority:")
    print(results_df.groupby("priority")["total_output_tokens"].mean())


USAGE SUMMARY
  Input tokens usage:      40119.333
  Output tokens usage:  709.333
Average score by Language:
language
de    46268.0
en    37045.0
Name: total_input_tokens, dtype: float64
Average score by Priority:
priority
medium    709.333333
Name: total_output_tokens, dtype: float64


# Close session 

In [6]:
session.close()